Write code to get the actual audio file for the corresponding M-DJCUE tracks (through youtube)

In [9]:
#script that goes through annotations and finds youtube links for the songs. 
import os
from pathlib import Path
import json

directory_path = Path("/Users/romanos/Documents/BeatBot/data/M-DJCUE/annotations")

tracks = []

for file_path in directory_path.iterdir():

    if file_path.is_file():
        with open(file_path, 'r') as f:
            data_dict = json.load(f)
        
        tracks.append(data_dict)

print(len(tracks))


134


Grab audio files from Youtube links......

In [8]:
import json
from pathlib import Path
import yt_dlp
import shutil

# Paths
directory_path = Path("/Users/romanos/Documents/BeatBot/data/M-DJCUE/annotations")
output_path = Path("/Users/romanos/Documents/BeatBot/data/M-DJCUE/audio")
output_path.mkdir(exist_ok=True)

# Find ffmpeg location
ffmpeg_location = shutil.which('ffmpeg')
if not ffmpeg_location:
    print("ERROR: FFmpeg not found in PATH")
    exit(1)

print(f"Using FFmpeg at: {ffmpeg_location}\n")

# Lists to track downloads
tracks = []
failed_downloads = []
successful_downloads = []

# Load all JAMS files and extract YouTube URLs
for file_path in directory_path.iterdir():
    if file_path.is_file() and file_path.suffix == '.jams':
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        metadata = data.get('file_metadata', {})
        title = metadata.get('title', 'Unknown')
        artist = metadata.get('artist', 'Unknown')
        
        identifiers = metadata.get('identifiers', {})
        mirrors = identifiers.get('mirrors', [])
        
        youtube_url = None
        for mirror in mirrors:
            if mirror.get('type') == 'Youtube':
                youtube_url = mirror.get('url')
                break
        
        if youtube_url:
            tracks.append({
                'artist': artist,
                'title': title,
                'url': youtube_url,
                'filename': f"{artist} - {title}"
            })

# Download audio with FFmpeg path specified
ydl_opts = {
    'format': 'bestaudio/best',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
    'outtmpl': str(output_path / '%(title)s.%(ext)s'),
    'ffmpeg_location': str(Path(ffmpeg_location).parent),  # Directory containing ffmpeg
    'extractor_args': {
        'youtube': {
            'player_client': ['android'],
            'skip': ['hls', 'dash']
        }
    },
    'quiet': True,
    'no_warnings': False,
}

print(f"Found {len(tracks)} tracks to download\n")

# Download each track
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    for i, track in enumerate(tracks, 1):
        try:
            print(f"[{i}/{len(tracks)}] Downloading: {track['artist']} - {track['title']}")
            ydl.download([track['url']])
            successful_downloads.append(track)
            print(f"✓ Success\n")
        except Exception as e:
            print(f"✗ Failed: {str(e)}\n")
            failed_downloads.append({
                'artist': track['artist'],
                'title': track['title'],
                'url': track['url'],
                'error': str(e)
            })

# Summary
print("=" * 80)
print(f"\nDownload Summary:")
print(f"Total tracks: {len(tracks)}")
print(f"Successful: {len(successful_downloads)}")
print(f"Failed: {len(failed_downloads)}")

# Print and save failed downloads
if failed_downloads:
    print(f"\n{'='*80}")
    print("Failed Downloads:")
    print("=" * 80)
    for item in failed_downloads:
        print(f"\nArtist: {item['artist']}")
        print(f"Title: {item['title']}")
        print(f"URL: {item['url']}")
        print(f"Error: {item['error']}")
    
    # Save failed downloads to JSON file
    failed_file = output_path / "failed_downloads.json"
    with open(failed_file, 'w') as f:
        json.dump(failed_downloads, f, indent=2)
    print(f"\n✓ Failed downloads list saved to: {failed_file}")
else:
    print("\n✓ All downloads successful!")


Using FFmpeg at: /opt/homebrew/bin/ffmpeg

Found 81 tracks to download

[1/81] Downloading: Elastic Reality - Cassa de X (Deep Dish Chamber Of Sound dub)


✓ Success                                                  

[2/81] Downloading: Freefall - Feel Surreal (12-inch Mix)


✓ Success                                                

[3/81] Downloading: Kings Of Tomorrow - Fall For You (Sandy Rivera's Classic Mix)


✓ Success                                                  

[4/81] Downloading: Kerri Chandler - Atmosphere EP (Track 1)


✓ Success                                                  

[5/81] Downloading: Fingers Inc - Can You Feel It (Instrumental)


...

✓ Success                                                  

[79/81] Downloading: Donna Summer - State of Independence (Murk Club Mix)


ERROR: [youtube] vMB5DRJYWEI: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.


✗ Failed: ERROR: [youtube] vMB5DRJYWEI: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.

[80/81] Downloading: Manu Dibango vs Masters At Work - New Bell


✓ Success                                                  

[81/81] Downloading: Caucasian Boy - Northern Lights (The Prime Cutz Mix)


✓ Success                                                  


Download Summary:
Total tracks: 81
Successful: 67
Failed: 14

Failed Downloads:

Artist: Fingers Inc
Title: Can You Feel It (Instrumental)
URL: https://www.youtube.com/watch?v=-gpuXqevQOU
Error: ERROR: [youtube] -gpuXqevQOU: Video unavailable

Artist: Karizma
Title: Tech This Out (Kaytronik Dub)
URL: https://www.youtube.com/watch?v=tP_R2IdqoB8
Error: ERROR: [youtube] tP_R2IdqoB8: Video unavailable. This video is not available

Artist: Colonel Abrams
Title: Victim Of Loving You (Vocal Club Mix)
URL: https://www.youtube.com/watch?v=UUWbLuaRe9A
Error: ERROR: [youtube] UUWbLuaRe9A: Video unavailable. This video is not available

Artist: Celetia
Title: Rewind (Ignorants Remix)
URL: https://www.youtube.com/watch?v=t-9oQPPdVKA
Error: ERROR: [youtube] t-9oQPPdVKA: Video unavailable

Artist: Nic Fanciulli
Title: Lucky Heather (Dubfire's Lucky 13 Remix
URL: https://www.youtube.com/watch?v=JX1BvUXupdc
Error: ERROR: [youtube] JX1BvUXupd

failed downloads:

[{'artist': 'Fingers Inc', 'title': 'Can You Feel It (Instrumental)', 'url': 'https://www.youtube.com/watch?v=-gpuXqevQOU', 'error': 'ERROR: [youtube] -gpuXqevQOU: Video unavailable'}, {'artist': 'Karizma', 'title': 'Tech This Out (Kaytronik Dub)', 'url': 'https://www.youtube.com/watch?v=tP_R2IdqoB8', 'error': 'ERROR: [youtube] tP_R2IdqoB8: Video unavailable. This video is not available'}, {'artist': 'Colonel Abrams', 'title': 'Victim Of Loving You (Vocal Club Mix)', 'url': 'https://www.youtube.com/watch?v=UUWbLuaRe9A', 'error': 'ERROR: [youtube] UUWbLuaRe9A: Video unavailable. This video is not available'}, {'artist': 'Celetia', 'title': 'Rewind (Ignorants Remix)', 'url': 'https://www.youtube.com/watch?v=t-9oQPPdVKA', 'error': 'ERROR: [youtube] t-9oQPPdVKA: Video unavailable'}, {'artist': 'Nic Fanciulli', 'title': "Lucky Heather (Dubfire's Lucky 13 Remix", 'url': 'https://www.youtube.com/watch?v=JX1BvUXupdc', 'error': 'ERROR: [youtube] JX1BvUXupdc: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'}, {'artist': 'Peven Everett', 'title': "I Can't Believe I Loved Her (Main Mix)", 'url': 'https://www.youtube.com/watch?v=5DtnLLbHnNw', 'error': 'ERROR: [youtube] 5DtnLLbHnNw: Video unavailable. This video is not available'}, {'artist': 'Joint Venture', 'title': 'Master Blaster (Turn It Up) (Tha Wild Pitch mix)', 'url': 'https://www.youtube.com/watch?v=GPkiGnPIfLw', 'error': 'ERROR: [youtube] GPkiGnPIfLw: Video unavailable. This video is not available'}, {'artist': 'River Ocean ft India', 'title': 'Conga Drums', 'url': 'https://www.youtube.com/watch?v=L6iqksHMrSM', 'error': 'ERROR: [youtube] L6iqksHMrSM: Video unavailable'}, {'artist': 'Future Sound Of London', 'title': 'Papua New Guinea (Dali Mix)', 'url': 'https://www.youtube.com/watch?v=Oyg8LFKI0KQ', 'error': 'ERROR: [youtube] Oyg8LFKI0KQ: Video unavailable. This video has been removed by the uploader'}, {'artist': 'Detlef', 'title': 'Kinky Tail', 'url': 'https://www.youtube.com/watch?v=zwOmaF3ZRdQ', 'error': 'ERROR: [youtube] zwOmaF3ZRdQ: Video unavailable'}, {'artist': 'Degrees Of Motion', 'title': 'Do You Want It Right Now (Extended Club Mix)', 'url': 'https://www.youtube.com/watch?v=mod5DGp4CDo', 'error': 'ERROR: [youtube] mod5DGp4CDo: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'}, {'artist': 'Sonique', 'title': 'Feels So Good (En-Motion Remix)', 'url': 'https://www.youtube.com/watch?v=b58-_6jU5qQ', 'error': 'ERROR: [youtube] b58-_6jU5qQ: Video unavailable'}, {'artist': "Mone'", 'title': 'We Can Make It (Joe T Vannelli Dub)', 'url': 'https://www.youtube.com/watch?v=cN1WyriwhEg', 'error': 'ERROR: [youtube] cN1WyriwhEg: Video unavailable. This video is not available'}, {'artist': 'Donna Summer', 'title': 'State of Independence (Murk Club Mix)', 'url': 'https://www.youtube.com/watch?v=vMB5DRJYWEI', 'error': 'ERROR: [youtube] vMB5DRJYWEI: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'}]

In [12]:
import json
from pathlib import Path

# Paths
directory_path = Path("/Users/romanos/Documents/BeatBot/data/M-DJCUE/annotations")
output_path = Path("/Users/romanos/Documents/BeatBot/data/M-DJCUE/audio")
failed_file = output_path / "failed_downloads.json"

# List to store all missing tracks
missing_tracks = []

# Load failed downloads if file exists
if failed_file.exists():
    with open(failed_file, 'r') as f:
        failed_downloads = json.load(f)
        for item in failed_downloads:
            missing_tracks.append({
                'artist': item['artist'],
                'title': item['title'],
                'url': item.get('url', None),
                'reason': 'download_failed',
                'error': item.get('error', '')
            })

# Find tracks with no YouTube mirrors
for file_path in directory_path.iterdir():
    if file_path.is_file() and file_path.suffix == '.jams':
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        metadata = data.get('file_metadata', {})
        title = metadata.get('title', 'Unknown')
        artist = metadata.get('artist', 'Unknown')
        
        identifiers = metadata.get('identifiers', {})
        mirrors = identifiers.get('mirrors', [])
        
        # Check if there's a YouTube URL
        youtube_url = None
        for mirror in mirrors:
            if mirror.get('type') == 'Youtube':
                youtube_url = mirror.get('url')
                break
        
        # If no YouTube URL found, add to missing list
        if not youtube_url:
            missing_tracks.append({
                'artist': artist,
                'title': title,
                'url': None,
                'reason': 'no_youtube_mirror',
                'error': ''
            })

# Print summary
print(f"Total missing tracks: {len(missing_tracks)}")
print(f"Failed downloads: {sum(1 for t in missing_tracks if t['reason'] == 'download_failed')}")
print(f"No YouTube mirror: {sum(1 for t in missing_tracks if t['reason'] == 'no_youtube_mirror')}")
print("\nMissing tracks list:")
print("=" * 80)

for track in missing_tracks:
    print(f"\nArtist: {track['artist']}")
    print(f"Title: {track['title']}")
    print(f"Reason: {track['reason']}")
    if track['url']:
        print(f"URL: {track['url']}")
    if track['error']:
        print(f"Error: {track['error']}")

# Save to file
missing_file = output_path / "missing_tracks.json"
with open(missing_file, 'w') as f:
    json.dump(missing_tracks, f, indent=2)

print(f"\n✓ Missing tracks list saved to: {missing_file}")

print(missing_tracks)

Total missing tracks: 67
Failed downloads: 14
No YouTube mirror: 53

Missing tracks list:

Artist: Fingers Inc
Title: Can You Feel It (Instrumental)
Reason: download_failed
URL: https://www.youtube.com/watch?v=-gpuXqevQOU
Error: ERROR: [youtube] -gpuXqevQOU: Video unavailable

Artist: Karizma
Title: Tech This Out (Kaytronik Dub)
Reason: download_failed
URL: https://www.youtube.com/watch?v=tP_R2IdqoB8
Error: ERROR: [youtube] tP_R2IdqoB8: Video unavailable. This video is not available

Artist: Colonel Abrams
Title: Victim Of Loving You (Vocal Club Mix)
Reason: download_failed
URL: https://www.youtube.com/watch?v=UUWbLuaRe9A
Error: ERROR: [youtube] UUWbLuaRe9A: Video unavailable. This video is not available

Artist: Celetia
Title: Rewind (Ignorants Remix)
Reason: download_failed
URL: https://www.youtube.com/watch?v=t-9oQPPdVKA
Error: ERROR: [youtube] t-9oQPPdVKA: Video unavailable

Artist: Nic Fanciulli
Title: Lucky Heather (Dubfire's Lucky 13 Remix
Reason: download_failed
URL: https://ww

All missing tracks:

[{'artist': 'Fingers Inc', 'title': 'Can You Feel It (Instrumental)', 'url': 'https://www.youtube.com/watch?v=-gpuXqevQOU', 'reason': 'download_failed', 'error': 'ERROR: [youtube] -gpuXqevQOU: Video unavailable'}, {'artist': 'Karizma', 'title': 'Tech This Out (Kaytronik Dub)', 'url': 'https://www.youtube.com/watch?v=tP_R2IdqoB8', 'reason': 'download_failed', 'error': 'ERROR: [youtube] tP_R2IdqoB8: Video unavailable. This video is not available'}, {'artist': 'Colonel Abrams', 'title': 'Victim Of Loving You (Vocal Club Mix)', 'url': 'https://www.youtube.com/watch?v=UUWbLuaRe9A', 'reason': 'download_failed', 'error': 'ERROR: [youtube] UUWbLuaRe9A: Video unavailable. This video is not available'}, {'artist': 'Celetia', 'title': 'Rewind (Ignorants Remix)', 'url': 'https://www.youtube.com/watch?v=t-9oQPPdVKA', 'reason': 'download_failed', 'error': 'ERROR: [youtube] t-9oQPPdVKA: Video unavailable'}, {'artist': 'Nic Fanciulli', 'title': "Lucky Heather (Dubfire's Lucky 13 Remix", 'url': 'https://www.youtube.com/watch?v=JX1BvUXupdc', 'reason': 'download_failed', 'error': 'ERROR: [youtube] JX1BvUXupdc: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'}, {'artist': 'Peven Everett', 'title': "I Can't Believe I Loved Her (Main Mix)", 'url': 'https://www.youtube.com/watch?v=5DtnLLbHnNw', 'reason': 'download_failed', 'error': 'ERROR: [youtube] 5DtnLLbHnNw: Video unavailable. This video is not available'}, {'artist': 'Joint Venture', 'title': 'Master Blaster (Turn It Up) (Tha Wild Pitch mix)', 'url': 'https://www.youtube.com/watch?v=GPkiGnPIfLw', 'reason': 'download_failed', 'error': 'ERROR: [youtube] GPkiGnPIfLw: Video unavailable. This video is not available'}, {'artist': 'River Ocean ft India', 'title': 'Conga Drums', 'url': 'https://www.youtube.com/watch?v=L6iqksHMrSM', 'reason': 'download_failed', 'error': 'ERROR: [youtube] L6iqksHMrSM: Video unavailable'}, {'artist': 'Future Sound Of London', 'title': 'Papua New Guinea (Dali Mix)', 'url': 'https://www.youtube.com/watch?v=Oyg8LFKI0KQ', 'reason': 'download_failed', 'error': 'ERROR: [youtube] Oyg8LFKI0KQ: Video unavailable. This video has been removed by the uploader'}, {'artist': 'Detlef', 'title': 'Kinky Tail', 'url': 'https://www.youtube.com/watch?v=zwOmaF3ZRdQ', 'reason': 'download_failed', 'error': 'ERROR: [youtube] zwOmaF3ZRdQ: Video unavailable'}, {'artist': 'Degrees Of Motion', 'title': 'Do You Want It Right Now (Extended Club Mix)', 'url': 'https://www.youtube.com/watch?v=mod5DGp4CDo', 'reason': 'download_failed', 'error': 'ERROR: [youtube] mod5DGp4CDo: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'}, {'artist': 'Sonique', 'title': 'Feels So Good (En-Motion Remix)', 'url': 'https://www.youtube.com/watch?v=b58-_6jU5qQ', 'reason': 'download_failed', 'error': 'ERROR: [youtube] b58-_6jU5qQ: Video unavailable'}, {'artist': "Mone'", 'title': 'We Can Make It (Joe T Vannelli Dub)', 'url': 'https://www.youtube.com/watch?v=cN1WyriwhEg', 'reason': 'download_failed', 'error': 'ERROR: [youtube] cN1WyriwhEg: Video unavailable. This video is not available'}, {'artist': 'Donna Summer', 'title': 'State of Independence (Murk Club Mix)', 'url': 'https://www.youtube.com/watch?v=vMB5DRJYWEI', 'reason': 'download_failed', 'error': 'ERROR: [youtube] vMB5DRJYWEI: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'}, {'artist': 'Butch Hohberg', 'title': 'Peyote', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Danny Tenaglia', 'title': 'Dibiza (Zoltan Kontes Jerome Robins Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'BeBe Winans', 'title': 'Thank You (KenLou Horn Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'D.Funk Era', 'title': '\u200eSpring', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Backroom Productions', 'title': 'Trouble (Dub Version 3)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Bent', 'title': 'Always (Mighty Mouse Remix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Carolyn Harding & Damon Horton', 'title': "Sing A Song (MAW's Instrumental)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Underworld', 'title': 'Beautiful Burnout', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Frankie Knuckles ft Adeva', 'title': "Too Many Fish (Morales' D-Bert Mix)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Urban Blues Project pres. Michael Procter', 'title': "Love Don't Live (U.B.P. Classic Mix)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'James Howard', 'title': 'We Can Do It (Wake Up)(The Wake Up Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Dusty Kid', 'title': "Singal'63", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Nalin & Kane', 'title': 'Beachball (Extended Version)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Black Coffee ft Thiwe', 'title': 'Crazy (Manoo & Francois A Deep Journey Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'U2', 'title': 'Lemon (Momo Beats)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Armand Van Helden', 'title': 'The Funk Phenomena', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Mike Delgado', 'title': "Byrdman's Revenge (Original Mix)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'MCJ ft Davina', 'title': "I'm Ready (For Your Love) (The Get Ready mix)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Deepswing', 'title': 'In The Music (Original Version)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': "Mang'e Le Funk", 'title': 'I Still Want You', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Max Berlin & Cicada', 'title': 'Elle Et Moi', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Bob Sinclair', 'title': 'Vision of Paradise', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Harry Romero ft Robert Owens', 'title': 'I Go Back', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Caucasian Boy', 'title': 'Northern Lights (Old Skool Re-edit)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Voices', 'title': 'Voices In My Mind (MAW Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Urban Blues Project pres. Michael Procter', 'title': "Love Don't Live (New Birth (Mix) UK Re-Edit)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'India', 'title': 'La India Con Lavoe (Remixed by MAW)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': '808 State', 'title': 'Pacific (Original 12-inch Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Fingers Inc', 'title': 'Can You Feel It (Original Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'C.J. Bolland', 'title': "Sugar Is Sweeter (Armand's Drum 'n' Bass Mix)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Liberty City', 'title': 'Some Lovin (Original Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Alistair Colling', 'title': 'Cafe Sol (Para Carolina) (Original)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Black Science Orchestra', 'title': "Keep On Keepin' On (Spen & Karisma's Deepah Dub)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Gerideau', 'title': 'Take A Stand (Original', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Danny Tenaglia', 'title': 'Elements (Original Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': '32 Project', 'title': 'Rule Of Humanity (Bonus Beats)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Jose Nunez', 'title': 'Bilingual (Dirty Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Cheek', 'title': 'Venus (Sunshine People Remix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Andy Ash', 'title': 'Hip Joint (Deep Space Orchestra Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Paperclip People', 'title': 'Remake Uno', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Incognito', 'title': 'Always There', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Barbara Tucker', 'title': "Stay Together (Armand's Crazy Trauma Mix)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Danny Tenaglia', 'title': 'The Better Days', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'LFO', 'title': 'LFO', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Rachid', 'title': "Pride (Dobie's It's A Thing Mix)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Fonda Rae', 'title': 'Living In Ecstasy (I Like What You Do) (The Groove Mix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Karen Pollack', 'title': 'You Cant Touch Me You Cant Hurt Me (Murk Remix)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'House Of Gypsies', 'title': "Samba (Tee's Freeze Mix)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Frankie Knuckles ft Adeva', 'title': 'Too Many Fish (Classic Frankie Version)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Don Carlos', 'title': 'Alone (Saxambient)', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Photon Inc. & Paula Brion', 'title': 'Generate Power (Wild Pitch Mix', 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Apparat', 'title': "Fractales (Apparat's Ibiza Mix", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}, {'artist': 'Satoshi Tomiie', 'title': "Tears (Full Intention's Drippin' 'n' Droppin' Dub)", 'url': None, 'reason': 'no_youtube_mirror', 'error': ''}]
